# AIML-02 training on Kaggle (free GPU)

1. Kaggle → *Create* → *New Notebook* → *File* → *Import Notebook* → this file.
2. *Add Input* → competition **rsna-pneumonia-detection-challenge** (accept the rules first).
3. *Settings* → Accelerator **GPU T4 x2** (or P100), Internet **On**.
4. *Add-ons* → *Secrets*: `WANDB_API_KEY` (optional but recommended), `GITHUB_REPO` = your repo URL.
5. *Run All*. Download `models/` and `reports/` from the Output tab when it finishes.

Everything here runs the **same DVC pipeline** as your laptop, so results are reproducible anywhere.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
repo = secrets.get_secret('GITHUB_REPO')
try:
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
except Exception:
    print('no W&B key: training will still run, just without live dashboards')
!git clone --depth 1 {repo} /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q 'monai>=1.4' nibabel 'pydicom>=3.0' 'dvc>=3.55' wandb && pip install -q --no-deps -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Point the pipeline at the read-only Kaggle input instead of copying 4 GB.
!mkdir -p data/raw && ln -sfn /kaggle/input/rsna-pneumonia-detection-challenge data/raw/rsna
!ls data/raw/rsna | head

In [ ]:
# Classification: prepare -> train -> evaluate (about 1.5-2 h on a T4)
!dvc repro eval_cls --force 2>&1 | tail -60

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open('reports/metrics.json')), indent=2))
pd.read_csv('reports/subgroups.csv')

In [ ]:
# 3D segmentation (spleen CT, downloads 1.5 GB from the MSD mirror; about 1.5 h on a T4).
# Run this in a SEPARATE Kaggle session if the classification run used most of the 12 h limit.
!dvc repro train_seg --force 2>&1 | tail -30

In [ ]:
# Bundle everything you need to bring back to the laptop.
!rm -rf data/raw && tar czf /kaggle/working/aiml02_outputs.tgz models reports dvc.lock
!ls -lh /kaggle/working/aiml02_outputs.tgz